#week13 / movie_review



컴퓨터공학과 / 202433638 / 장영환

IMDB 영화 리뷰 데이터셋을 사용해서 영화 리뷰가 긍정 리뷰인지 부정 리뷰인지 분류하는 이진 분류 모델

In [26]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

자주 등장하는 상위 10,000개 단어만 사용

In [27]:
imdb = keras.datasets.imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words = 10000)

In [28]:
len(x_train), len(x_test)

(25000, 25000)

In [29]:
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


IMDB 데이터셋의 리뷰가 이미 문자열이 아니라 정수 시퀀스(sequence of integers) 로 저장됨 정수는 단어의 ID에 해당

In [30]:
len(x_train[0]), len(x_test[0])

(218, 68)

리뷰마다 길이가 다르다.

In [31]:
y_train[0], y_train[1]

(np.int64(1), np.int64(0))

첫 번째 학습 리뷰: 긍정
두 번째 학습 리뷰: 부정

라벨 분포 확인

In [32]:
np.unique(y_train, return_counts = True)

(array([0, 1]), array([12500, 12500]))

In [33]:
# 단어 ->정수 인덱스 딕셔너리
word_to_index = imdb.get_word_index()

# 처음 몇 개의 인덱스는 특수 용도로 사용된다.
word_to_index = {k:(v+3) for k,v in word_to_index.items()}
word_to_index["<PAD>"] = 0 # 문장을 채우는 기호
word_to_index["<START>"] = 1 # 시작을 표시
word_to_index["<UNK>"] = 2 # 알려지지 않은 토큰
word_to_index["<UNUSED>"] = 3

index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

원래 단어 인덱스에 +3, 앞쪽 번호를 특수 토큰으로 사용

word_to_index = imdb.get_word_index() 단어를 정수 인덱스로 바꾸는 딕셔너리

모든 단어가 실제 단어는 아니고, 모델 입력을 안정적으로 처리하기 위한 특수 토큰(special token) 이 필요

딕셔너리를 반대로 뒤집는다

word_to_index:
단어 → 정수

index_to_word:
정수 → 단어

In [34]:
print(' '.join([index_to_word[index] for index in x_train[0]]))

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha

정수로 저장된 리뷰를 다시 사람이 읽을 수 있는 단어 문자열로 복원

정수 시퀀스가 실제로 어떤 문장을 의미하는지 확인

In [35]:
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import *

In [36]:
x_train = pad_sequences(x_train, maxlen = 100)
x_test = pad_sequences(x_test, maxlen = 100)

모든 리뷰를 길이 100으로

In [37]:
len(x_train[0]), len(x_train[1])

(100, 100)

In [38]:
print(x_train[0])

[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


In [39]:
vocab_size = 10000
model = Sequential()
model.add(Embedding(vocab_size, 64,
input_length=100))
model.add(Flatten())
model.add(Dense(64, activation = 'relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.compile(loss = 'binary_crossentropy', optimizer='adam',
metrics=['accuracy'])
history = model.fit(x_train, y_train,
batch_size = 64, epochs=20, verbose=1,
validation_data=(x_test, y_test))

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 30ms/step - accuracy: 0.7563 - loss: 0.4696 - val_accuracy: 0.8231 - val_loss: 0.4321
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.9351 - loss: 0.1797 - val_accuracy: 0.8285 - val_loss: 0.4172
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.9923 - loss: 0.0327 - val_accuracy: 0.8346 - val_loss: 0.5232
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9989 - loss: 0.0067 - val_accuracy: 0.8348 - val_loss: 0.6054
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.9998 - loss: 0.0022 - val_accuracy: 0.8390 - val_loss: 0.6551
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.9998 - loss: 0.0015 - val_accuracy: 0.8369 - val_loss: 0.7067
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9998 - loss: 0.0011 - val_accuracy: 0.8385 - val_loss: 0.7293
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 1.0000 - loss: 4.1523e-04 - 

In [41]:
results = model.evaluate(x_test, y_test, verbose=2)
print(results)

782/782 - 3s - 4ms/step - accuracy: 0.8242 - loss: 1.1508
[1.1508351564407349, 0.8242400288581848]


과적합발생 : 모델이 학습 데이터에 대해서는 거의 완벽하게 맞힌다.

train accuracy ≈ 1.0000

하지만 테스트 데이터에서는 다음 정도에 머문다.

test accuracy ≈ 0.8242

In [43]:
review = "What can I say about this movie that was already said? It is my favorite time travel sci-fi, adventure epic comedy in the 80's and I love this movie to death! When I saw this movie I was thrown out by its theme. An excellent sci-fi, adventure epic, I LOVE the 80s. It's simple the greatest time travel movie ever happened in the history of world cinema. I love this movie todeath, I love, LOVE, love it!"

In [45]:
import re
review = re.sub("[^0-9a-zA-Z ]", "", review).lower() #전처리

review_encoding = []
# 리뷰의 각 단어 대하여 반복한다.
for w in review.split():
		index = word_to_index.get(w, 2)	# 딕셔너리에 없으면 2 반환
		if index <= 10000:		# 단어의 개수는 10000이하
			review_encoding.append(index)
		else:
			review_encoding.append(word_to_index["UNK"])

# 2차원 리스트로 전달하여야 한다.
test_input = pad_sequences([review_encoding], maxlen = 100)
value = model.predict(test_input) # 예측
if(value > 0.5):
	print("긍정적인 리뷰입니다.")
else:
	print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
긍정적인 리뷰입니다.
